In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time
import random

In [2]:
# Set random seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Random seed set to {seed}")

set_seed(42)

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

DATA_ROOT = r"D:\Uni\Lab\inebriation-voice-detector\data\processed"
TRAIN_DIR = os.path.join(DATA_ROOT, "TRAIN")
VAL_DIR = os.path.join(DATA_ROOT, "VALIDATION")
TEST_DIR = os.path.join(DATA_ROOT, "TEST")

# Create output directory
os.makedirs("output", exist_ok=True)

# Define classes
classes = ['SOBER', 'DRUNK']
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
idx_to_class = {idx: cls for idx, cls in enumerate(classes)}

Random seed set to 42
Using device: cuda:0


In [3]:
#Dataset Class
class SpectrogramDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = ['SOBER', 'DRUNK']
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        self.samples = self._make_dataset()
        
    def _make_dataset(self):
        samples = []
        for class_name in os.listdir(self.root_dir):
            class_path = os.path.join(self.root_dir, class_name)
            if os.path.isdir(class_path):
                class_idx = self.class_to_idx[class_name]
                for filename in os.listdir(class_path):
                    if filename.endswith(('.jpg', '.jpeg', '.png')):
                        samples.append((
                            os.path.join(class_path, filename),
                            class_idx
                        ))
        return samples
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        return image, label



In [4]:
# Data Preparation
# Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Initialize datasets

train_dataset = SpectrogramDataset(TRAIN_DIR, transform)
val_dataset = SpectrogramDataset(VAL_DIR, transform)
test_dataset = SpectrogramDataset(TEST_DIR, transform)

# Create balanced sampler
class_counts = torch.tensor([len([x for x in train_dataset.samples if x[1] == i]) for i in range(2)])
class_weights = 1. / class_counts
sample_weights = torch.tensor([class_weights[y] for _, y in train_dataset.samples])
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

# Create data loaders
BATCH_SIZE = 100
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [5]:
#Model
def create_model():
    # Load pretrained ResNet18
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    
    # Freeze all layers except last block
    for name, param in model.named_parameters():
        if not name.startswith('layer4'):
            param.requires_grad = False
    
    # Replace final layer for binary classification
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 1),
        nn.Sigmoid()
    )
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"\nModel created:")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    
    return model.to(device)

model = create_model()


Model created:
Total parameters: 11,177,025
Trainable parameters: 8,394,241


In [6]:
#Loss Function
class WeightedBCELoss(nn.Module):
    """Weighted Binary Cross Entropy as per ADLAIA paper"""
    def __init__(self, weight_pos=0.9, weight_neg=0.1):
        super().__init__()
        self.weight_pos = weight_pos
        self.weight_neg = weight_neg
        
    def forward(self, inputs, targets):
        loss = - (self.weight_pos * targets * torch.log(inputs + 1e-7) + 
                 self.weight_neg * (1 - targets) * torch.log(1 - inputs + 1e-7))
        return torch.mean(loss)

criterion = WeightedBCELoss(weight_pos=0.9, weight_neg=0.1)
print("\nUsing weighted BCE loss with:")
print(f"Positive weight (DRUNK): 0.9")
print(f"Negative weight (SOBER): 0.1")




Using weighted BCE loss with:
Positive weight (DRUNK): 0.9
Negative weight (SOBER): 0.1


In [7]:
# Training Setup
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

print("\nTraining setup:")
print(f"Optimizer: Adam(lr=0.001)")
print(f"LR scheduler: StepLR(step_size=10, gamma=0.1)")

# Cell 9: Evaluation Function
def evaluate(model, data_loader):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    running_loss = 0.0
    
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).float()
            
            outputs = model(inputs).squeeze()
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            all_probs.extend(outputs.cpu().numpy())
            all_preds.extend((outputs >= 0.5).float().cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate metrics
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    # ROC curve
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)
    
    metrics = {
        'loss': running_loss / len(data_loader.dataset),
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, zero_division=0),
        'recall': recall_score(all_labels, all_preds, zero_division=0),
        'f1': f1_score(all_labels, all_preds, zero_division=0),
        'auc': roc_auc,
        'confusion_matrix': confusion_matrix(all_labels, all_preds)
    }
    
    return metrics




Training setup:
Optimizer: Adam(lr=0.001)
LR scheduler: StepLR(step_size=10, gamma=0.1)


In [8]:
#  Training Loop
def train_model(model, train_loader, val_loader, num_epochs=30):
    best_f1 = 0
    history = {'train': [], 'val': []}
    
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 10)
        
        # Train
        model.train()
        train_loss = 0
        for inputs, labels in tqdm(train_loader, desc="Training"):
            inputs, labels = inputs.to(device), labels.to(device).float()
            
            optimizer.zero_grad()
            outputs = model(inputs).squeeze()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
        
        # Validate
        val_metrics = evaluate(model, val_loader)
        train_loss = train_loss / len(train_loader.dataset)
        
        # Update LR
        scheduler.step()
        
        # Save history
        history['train'].append(train_loss)
        history['val'].append(val_metrics['loss'])
        
        # Print metrics
        print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_metrics['loss']:.4f}")
        print(f"Val Accuracy: {val_metrics['accuracy']:.4f}")
        print(f"Val F1: {val_metrics['f1']:.4f} | Val AUC: {val_metrics['auc']:.4f}")
        
        # Save best model
        if val_metrics['f1'] > best_f1:
            best_f1 = val_metrics['f1']
            torch.save(model.state_dict(), "output/best_model.pth")
            print("Saved new best model!")
    
    return history

# Train for 30 epochs as per paper
history = train_model(model, train_loader, val_loader, num_epochs=30)



Epoch 1/30
----------


Training: 100%|████████████████████████████████████████████████████████████████████████| 75/75 [01:19<00:00,  1.06s/it]


Train Loss: 0.1584 | Val Loss: 0.1870
Val Accuracy: 0.3662
Val F1: 0.4718 | Val AUC: 0.6436
Saved new best model!

Epoch 2/30
----------


Training: 100%|████████████████████████████████████████████████████████████████████████| 75/75 [01:26<00:00,  1.15s/it]


Train Loss: 0.1040 | Val Loss: 0.2488
Val Accuracy: 0.4057
Val F1: 0.4747 | Val AUC: 0.6215
Saved new best model!

Epoch 3/30
----------


...

Train Loss: 0.0003 | Val Loss: 0.7570
Val Accuracy: 0.6628
Val F1: 0.4632 | Val AUC: 0.6562

Epoch 29/30
----------


Training: 100%|████████████████████████████████████████████████████████████████████████| 75/75 [01:05<00:00,  1.15it/s]


Train Loss: 0.0005 | Val Loss: 0.7386
Val Accuracy: 0.6630
Val F1: 0.4715 | Val AUC: 0.6580

Epoch 30/30
----------


Training: 100%|████████████████████████████████████████████████████████████████████████| 75/75 [01:06<00:00,  1.14it/s]


Train Loss: 0.0005 | Val Loss: 0.7354
Val Accuracy: 0.6492
Val F1: 0.4686 | Val AUC: 0.6551


In [9]:

#  Test Evaluation
def plot_results(metrics, class_names):
    # Confusion matrix
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    sns.heatmap(metrics['confusion_matrix'], annot=True, fmt='d', 
                cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    
    # ROC curve
    plt.subplot(1, 2, 2)
    fpr, tpr, _ = roc_curve(metrics['true_labels'], metrics['probabilities'])
    plt.plot(fpr, tpr, label=f"AUC = {metrics['auc']:.2f}")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    
    plt.tight_layout()
    plt.savefig("output/test_results.png")
    plt.show()

# Load best model and evaluate
model.load_state_dict(torch.load("output/best_model.pth"))
test_metrics = evaluate(model, test_loader)

print("\nFinal Test Results:")
print(f"Accuracy: {test_metrics['accuracy']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall: {test_metrics['recall']:.4f}")
print(f"F1 Score: {test_metrics['f1']:.4f}")
print(f"AUC: {test_metrics['auc']:.4f}")



C:\Users\krevi\AppData\Local\Temp\ipykernel_18788\1276551505.py:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("output/best_model.pth"))



Final Test Results:
Accuracy: 0.5711
Precision: 0.5222
Recall: 0.7799
F1 Score: 0.6255
AUC: 0.6381

Saved final model to output/final_model.pth
